# Four-way factor benchmark — tonsil B cells (King et al. 2021, Sci. Immunol.)

Trains SemanticSCVI / LDVAE+ / scHPF / cNMF on `tonsil_b_clean.h5ad` (≈26k tonsil B cells: naive / GC (centroblast + centrocyte) / memory / plasmablast, 4 000 HVG, raw counts) and benchmarks against `lib1_immune.gmt` + `lib2_bcell.gmt`. Same pipeline as `four_way_benchmark.ipynb`; only the data paths + cache dirs differ.

All architecture / training knobs are explicit in **Cell 2 (parameters)** — edit there to sweep.

In [ ]:
# ============================================================
# Parameters — edit here. All training/architecture knobs in one place.
# ============================================================
from pathlib import Path

# Anchor every path to notebooks/. Works whether jupyter was launched from
# notebooks/, the project root, or a parent dir of the project.
def _find_nb_dir() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        for cand in (
            p / "notebooks" / "benchmark_helpers.py",
            p / "benchmark_helpers.py",
            p / "scvi-tools-neural-nmf" / "notebooks" / "benchmark_helpers.py",
        ):
            if cand.exists():
                return cand.parent.resolve()
    raise FileNotFoundError(
        f"Could not locate benchmark_helpers.py from cwd={Path.cwd()}. "
        "Launch jupyter under the scvi-tools-neural-nmf tree, or set NB_DIR manually."
    )

NB_DIR = _find_nb_dir()
print(f"NB_DIR = {NB_DIR}")

ADATA_PATH = NB_DIR / "tonsil_b_clean.h5ad"
SEMANTIC_CACHE = NB_DIR / "tonsil_b_clean_geneformer.pt"
PATHWAY_INDEX = NB_DIR / "pathway_index.pkl"
LIB1_GMT = NB_DIR / "lib1_immune.gmt"
LIB2_GMT = NB_DIR / "lib2_bcell.gmt"
OUT_DIR = NB_DIR / "benchmark_results" / "bcell_four_way"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Preprocessing ----
HVG_TOP_N = None       # None = use all genes; int = take top-N HVGs (sc.pp.highly_variable_genes)
HVG_FLAVOR = "seurat_v3"  # works on raw counts; switch to "seurat" / "cell_ranger" if you log-normalized

# ---- Cache / retrain control ----
MODEL_CACHE_DIR = NB_DIR / ".model_cache_bcell"
FORCE_TRAIN_SEMANTIC = True  # ← retrain SemanticSCVI even if cached
FORCE_TRAIN_LDVAE = False
FORCE_TRAIN_SCHPF = False
FORCE_TRAIN_NMF = False

# ---- Shared model size ----
N_LATENT = 10

# ---- SemanticSCVI training ----
# use_batch_norm flows into BOTH encoder and decoder on the upstream LDVAE.
# Set to False to drop batch norm from the linear decoder (and the encoder).
SEMANTIC_MAX_EPOCHS = 200
SEMANTIC_WARMUP_EPOCHS = 20
SEMANTIC_N_EPOCHS_KL_WARMUP = 40  # linear KL anneal 0→1 over N epochs; finishes shortly after semantic loss turns on
SEMANTIC_KWARGS = dict(
    loss_mode="geometric",
    coherence_weight=2000.0,
    n_gene_sample=1024,
    n_latent=N_LATENT,
    n_layers=2,
    n_hidden=128,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)

# ---- LDVAE+ training ----
LDVAE_MAX_EPOCHS = 250
LDVAE_KWARGS = dict(
    n_hidden=128,
    n_latent=N_LATENT,
    n_layers=1,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)

# ---- scHPF / cNMF ----
N_FACTORS = N_LATENT
NMF_INIT = "nndsvd"
NMF_MAX_ITER = 500
NMF_RANDOM_STATE = 42

# ---- Benchmark / report ----
MODEL_NAMES = ["semantic_geom", "ldvae_nn", "schpf_k10", "nmf_k10"]
PER_PROJECTION_N_TOP = 50
CLUSTER_N_TOP = 500     # pool of top-loaded genes fed into UMAP+Leiden and hierarchical clustering
GENE_MAPPING = ("feature_id", "feature_name")  # Ensembl → symbol via adata.var columns
ENABLE_LLM_GRADING = True  # set False to skip Sonnet/Haiku scoring

In [ ]:
import sys
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import scanpy as sc
import torch
from sklearn.decomposition import NMF
from scipy.sparse import coo_matrix, issparse
import numpy as np
import schpf

from benchmark_helpers import (
    NMFWrapper,
    _ScviAdapter,
    build_report,
    get_or_build_geneformer_map,
    train_or_load_nonneg_ldvae,
    train_or_load_pickle,
    train_or_load_semantic_scvi,
)
from benchmarking import SemanticBenchmark
from train_schpf import train_nmf_model, train_schpf_model

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"NB_DIR = {NB_DIR}")

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
print("adata (raw):", adata.shape)
print("var cols:", list(adata.var.columns))

# Build/load the FULL-gene Geneformer semantic map first, then subset along with adata.
semantic_map = get_or_build_geneformer_map(adata, SEMANTIC_CACHE)
print("semantic_map (raw):", tuple(semantic_map.shape))

if HVG_TOP_N is not None:
    sc.pp.highly_variable_genes(
        adata, n_top_genes=HVG_TOP_N, flavor=HVG_FLAVOR, subset=False
    )
    keep = adata.var["highly_variable"].values
    adata = adata[:, keep].copy()
    semantic_map = semantic_map[torch.as_tensor(keep)]
    print(f"After HVG filter (top {HVG_TOP_N}, flavor={HVG_FLAVOR}):")
    print("  adata:", adata.shape)
    print("  semantic_map:", tuple(semantic_map.shape))
else:
    print("HVG filter skipped (HVG_TOP_N=None)")

In [ ]:
# Each scvi model writes a UUID into adata.uns, so each needs its own copy.
adata_sem = adata.copy()
semantic_model = train_or_load_semantic_scvi(
    adata_sem,
    semantic_map,
    cache_dir=MODEL_CACHE_DIR / "semantic_scvi",
    force_train=FORCE_TRAIN_SEMANTIC,
    max_epochs=SEMANTIC_MAX_EPOCHS,
    warmup_epochs=SEMANTIC_WARMUP_EPOCHS,
    n_epochs_kl_warmup=SEMANTIC_N_EPOCHS_KL_WARMUP,
    **SEMANTIC_KWARGS,
)

adata_ldvae = adata.copy()
ldvae_model = train_or_load_nonneg_ldvae(
    adata_ldvae,
    cache_dir=MODEL_CACHE_DIR / "ldvae_nn",
    force_train=FORCE_TRAIN_LDVAE,
    max_epochs=LDVAE_MAX_EPOCHS,
    **LDVAE_KWARGS,
)

schpf_model = train_or_load_pickle(
    "scHPF",
    lambda: train_schpf_model(adata, n_factors=N_FACTORS),
    cache_path=MODEL_CACHE_DIR / "schpf_k10.pkl",
    force_train=FORCE_TRAIN_SCHPF,
)

def _train_nmf():
    print(f"Training cNMF (sklearn NMF, k={N_FACTORS}, init={NMF_INIT})...")
    X_nmf = adata.X
    if issparse(X_nmf):
        if (X_nmf.data < 0).any():
            X_nmf.data = np.maximum(X_nmf.data, 0)
    else:
        X_nmf = np.maximum(X_nmf, 0)
    _nmf = NMF(
        n_components=N_FACTORS,
        init=NMF_INIT,
        random_state=NMF_RANDOM_STATE,
        max_iter=NMF_MAX_ITER,
    )
    W = _nmf.fit_transform(X_nmf)
    H = _nmf.components_
    return NMFWrapper(model=_nmf, W=W, H=H, feature_names=adata.var_names)

nmf_model = train_or_load_pickle(
    "cNMF/NMF",
    _train_nmf,
    cache_path=MODEL_CACHE_DIR / "nmf_k10.pkl",
    force_train=FORCE_TRAIN_NMF,
)

In [ ]:
# ============================================================
# Training-loss curves — train/val for SemanticSCVI and LDVAE+.
# scHPF / cNMF have no per-epoch history, so they're skipped.
# ============================================================
import importlib, benchmark_helpers
importlib.reload(benchmark_helpers)
from benchmark_helpers import plot_training_curves

plot_training_curves(
    semantic_model, "SemanticSCVI",
    OUT_DIR / "training_curves_semantic.png",
    semantic=True,
)
plot_training_curves(
    ldvae_model, "LDVAE+",
    OUT_DIR / "training_curves_ldvae.png",
    semantic=False,
)
print("scHPF / cNMF: no per-epoch training history (skipping).")


In [ ]:
models = {
    MODEL_NAMES[0]: _ScviAdapter(semantic_model, adata_sem),
    MODEL_NAMES[1]: _ScviAdapter(ldvae_model, adata_ldvae),
    MODEL_NAMES[2]: schpf_model,
    MODEL_NAMES[3]: nmf_model,
}
for name, m in models.items():
    z = m.get_latent_representation()
    print(f"{name}: latent shape = {z.shape}")

In [ ]:
bench = SemanticBenchmark(
    models,
    adata,
    pathway_index=str(PATHWAY_INDEX),
    gene_mapping=GENE_MAPPING,
    out_dir=str(OUT_DIR),
)
bench.run_figures(
    semantic_map=semantic_map,
    lib1_gmt=str(LIB1_GMT) if LIB1_GMT.exists() else None,
    lib2_gmt=str(LIB2_GMT) if LIB2_GMT.exists() else None,
    per_projection_n_top=PER_PROJECTION_N_TOP,
    cluster_n_top=CLUSTER_N_TOP,
)

In [ ]:
if ENABLE_LLM_GRADING:
    bench.run_grading(
        lib1_gmt=str(LIB1_GMT) if LIB1_GMT.exists() else None,
        lib2_gmt=str(LIB2_GMT) if LIB2_GMT.exists() else None,
        per_projection_n_top=PER_PROJECTION_N_TOP,
        cluster_n_top=CLUSTER_N_TOP,
    )
else:
    print("LLM grading skipped (ENABLE_LLM_GRADING=False)")

In [ ]:
from datetime import datetime

notes = (
    f"SemanticSCVI: {SEMANTIC_KWARGS['loss_mode']} loss, "
    f"cw={SEMANTIC_KWARGS['coherence_weight']}, "
    f"warmup={SEMANTIC_WARMUP_EPOCHS}/{SEMANTIC_MAX_EPOCHS} ep. "
    f"LDVAE+: weights_positive=True, {LDVAE_MAX_EPOCHS} ep. "
    f"scHPF: k={N_FACTORS}. cNMF: sklearn NMF k={N_FACTORS}, init={NMF_INIT}."
)
report_path = build_report(OUT_DIR, MODEL_NAMES, adata.shape, notes=notes)

# Rename to <dataset>_<YYYYMMDD_HHMMSS>.html and drop intermediate artifacts.
# Preserved: LLM/score caches (`_llm_cache/`, `_score_cache/`) and prior .html reports.
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
final_name = f"{ADATA_PATH.stem}_{ts}.html"
final_path = OUT_DIR / final_name
report_path.rename(final_path)

PRESERVE_DIRS = {"_llm_cache", "_score_cache"}
for child in OUT_DIR.iterdir():
    if child == final_path:
        continue
    if child.is_dir():
        if child.name in PRESERVE_DIRS:
            continue
        import shutil
        shutil.rmtree(child)
    else:
        if child.suffix == ".html":
            continue
        child.unlink()

print(f"Report written: {final_path}")
print(f"Size: {final_path.stat().st_size / 1e6:.1f} MB")